In [3]:
import os
import cv2
import numpy as np

def create_dataset(video_dir, label, frame_count=10, frame_size=(256, 256)):
    videos = []
    labels = []
    i=0
    for video_file in os.listdir(video_dir):
        video_path = os.path.join(video_dir, video_file)
        cap = cv2.VideoCapture(video_path)
        frames = []
        i+=1
        print(i, end=" ")
        if i == 1000:
            break
        # Read the specified number of frames
        while len(frames) < frame_count:
            ret, frame = cap.read()
            if not ret:
                break
            # Resize frame
            frame = cv2.resize(frame, frame_size)
            frames.append(frame)
        
        cap.release()
        
        # Only add the video if it has the required number of frames
        if len(frames) == frame_count:
            videos.append(np.array(frames))  # Convert frames to NumPy array
            labels.append(label)
    
    return videos, labels  # Return as lists, not as NumPy arrays

# Example usage:
real_videos_dir = 'celeb-df-v2/Celeb-real'
fake_videos_dir = 'celeb-df-v2/Celeb-synthesis'

# Create datasets
real_videos, real_labels = create_dataset(real_videos_dir, label=1)
fake_videos, fake_labels = create_dataset(fake_videos_dir, label=0)

# Combine real and fake datasets
videos = real_videos + fake_videos
labels = real_labels + fake_labels

# Convert lists to NumPy arrays after combining
videos = np.array(videos)
labels = np.array(labels)

print(f"Shape of video dataset: {videos.shape}")
print(f"Shape of labels: {labels.shape}")

1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197 198 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215 216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233 234 235 236 237 238 239 240 241 242 243 244 245 246 247 248 249 250 251 252 253 254 255 256 257 258 259 260 261 262 263 264 265 266 267 268 269 270 271 272 273 274 275 276 277 

In [14]:
print(videos[0].shape)

torch.Size([3, 10, 256, 256])


In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np


class DeepFakeDetectionModel(nn.Module):
    def __init__(self, input_shape):
        super(DeepFakeDetectionModel, self).__init__()

        # Unpack input shape
        frames, channels, height, width = input_shape

        # Convolutional layers
        self.features = nn.Sequential(
            # Conv3D Layer 1
            nn.Conv3d(channels, 8, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.ReLU(inplace=True),
            nn.BatchNorm3d(8),
            nn.MaxPool3d((2, 2, 2)),

            # Conv3D Layer 2
            nn.Conv3d(8, 16, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.ReLU(inplace=True),
            nn.BatchNorm3d(16),
            nn.MaxPool3d((2, 2, 2)),

            # Conv3D Layer 3
            nn.Conv3d(16, 32, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.ReLU(inplace=True),
            nn.BatchNorm3d(32),
            nn.MaxPool3d((2, 2, 2))
        )

        # Calculate the flattened size
        with torch.no_grad():
            test_input = torch.zeros(1, *input_shape)
            flattened_size = self.features(test_input).view(1, -1).size(1)

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(flattened_size, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # Ensure input is in correct shape
        # PyTorch Conv3d expects input: [batch_size, channels, frames, height, width]
        # If input is [batch_size, frames, channels, height, width], we need to permute
        if x.dim() == 5 and x.size(1) > x.size(2):
            x = x.permute(0, 2, 1, 3, 4).contiguous()

        features = self.features(x)
        features = features.view(features.size(0), -1)
        output = self.classifier(features)
        return output


class VideoDataset(Dataset):
    def __init__(self, videos, labels):
        # Ensure videos are in the right format: [batch, channels, frames, height, width]
        if isinstance(videos, np.ndarray):
            # Convert numpy array to torch tensor
            self.videos = torch.from_numpy(videos).float()

            # Permute if necessary (from [batch, frames, channels, height, width] to [batch, channels, frames, height, width])
            if self.videos.dim() == 5 and self.videos.size(1) > self.videos.size(2):
                self.videos = self.videos.permute(0, 2, 1, 3, 4).contiguous()
        else:
            # If already a tensor, ensure it's float
            self.videos = videos.float()

        # Convert labels to float for binary classification
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.videos)

    def __getitem__(self, idx):
        return self.videos[idx], self.labels[idx]


class DeepFakeDetector:
    def __init__(self, input_shape):
        self.device = torch.device(
            'cuda' if torch.cuda.is_available() else 'cpu')
        self.model = DeepFakeDetectionModel(input_shape).to(self.device)
        self.criterion = nn.BCELoss()
        self.optimizer = optim.Adam(self.model.parameters())

    def train(self, train_data, validation_data, epochs=28, batch_size=16):
        # Create DataLoaders
        train_loader = DataLoader(
            train_data, batch_size=batch_size, shuffle=True)
        validation_loader = DataLoader(
            validation_data, batch_size=batch_size, shuffle=False)

        # Training loop
        for epoch in range(epochs):
            # Training phase
            self.model.train()
            train_loss = 0.0
            for videos, labels in train_loader:
                videos, labels = videos.to(self.device), labels.to(self.device)

                # Zero the parameter gradients
                self.optimizer.zero_grad()

                # Forward pass
                outputs = self.model(videos)
                loss = self.criterion(outputs.squeeze(), labels)

                # Backward pass and optimize
                loss.backward()
                self.optimizer.step()

                train_loss += loss.item()

            # Validation phase
            self.model.eval()
            val_loss = 0.0
            correct = 0
            total = 0
            with torch.no_grad():
                for videos, labels in validation_loader:
                    videos, labels = videos.to(
                        self.device), labels.to(self.device)

                    outputs = self.model(videos)
                    loss = self.criterion(outputs.squeeze(), labels)
                    val_loss += loss.item()

                    # Calculate accuracy
                    predicted = (outputs.squeeze() > 0.5).float()
                    total += labels.size(0)
                    correct += (predicted == labels).sum().item()

            # Print epoch statistics
            print(f"Epoch [{epoch+1}/{epochs}]")
            print(f"Train Loss: {train_loss/len(train_loader):.4f}")
            print(f"Validation Loss: {val_loss/len(validation_loader):.4f}")
            print(f"Validation Accuracy: {100 * correct / total:.2f}%")

    def save(self, filepath):
        torch.save(self.model.state_dict(), filepath)

    def load(self, filepath):
        self.model.load_state_dict(torch.load(filepath))

    def predict(self, data):
        self.model.eval()
        with torch.no_grad():
            # Ensure data is in the right shape
            if isinstance(data, np.ndarray):
                data = torch.from_numpy(data).float()

            # Permute if necessary
            if data.dim() == 5 and data.size(1) > data.size(2):
                data = data.permute(0, 2, 1, 3, 4).contiguous()

            data = data.to(self.device)
            outputs = self.model(data)
            return outputs.cpu().numpy()


# Example usage
# Define input shape
frame_count = 10
height = width = 256
channels = 3
input_shape = (frame_count, channels, height, width)  # (10, 3, 256, 256)

# Example of how to use with potentially misshapen input
# videos: shape [batch_size, 10, 3, 256, 256] or [batch_size, 3, 10, 256, 256]
# labels: shape [batch_size]
videos = torch.randn(32, 10, 3, 256, 256)  # example random input
labels = torch.randint(0, 2, (32,)).float()  # random binary labels

# Ensure videos are in the correct shape: [batch_size, channels, frames, height, width]
if videos.size(1) > videos.size(2):
    videos = videos.permute(0, 2, 1, 3, 4).contiguous()

train_data = VideoDataset(videos, labels)
validation_data = VideoDataset(videos, labels)

deepfake_detector = DeepFakeDetector(input_shape)

# Train the model
deepfake_detector.train(train_data, validation_data, epochs=28)

C:\Users\aaron\AppData\Local\Temp\ipykernel_3880\1059902777.py:78: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.labels = torch.tensor(labels, dtype=torch.float32)


RuntimeError: Given groups=1, weight of size [8, 3, 3, 3, 3], expected input[1, 10, 3, 256, 256] to have 3 channels, but got 10 channels instead

TensorFlow GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


Physical devices cannot be modified after being initialized
Model: "DeepFakeDetectionModel"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_3 (InputLayer)        [(None, 10, 256, 256, 3)  0         
                             ]                                   
                                                                 
 conv3d_1 (Conv3D)           (None, 10, 256, 256, 8)   656       
                                                                 
 batch_normalization_1 (Batc  (None, 10, 256, 256, 8)  32        
 hNormalization)                                                 
                                                                 
 max_pooling3d_1 (MaxPooling  (None, 5, 128, 128, 8)   0         
 3D)                                                             
                                                                 
 conv3d_2 (Conv3D)           (None, 5, 128, 128, 1

InternalError: Failed copying input tensor from /job:localhost/replica:0/task:0/device:CPU:0 to /job:localhost/replica:0/task:0/device:GPU:0 in order to run _EagerConst: Dst tensor is not initialized.